### Goal of this notebook 

Practice running Nextflow pipelines locally on a SageMaker notebook instance.

### What Works ✅

- **Nextflow installation**: Version 24.04.4 installed via conda
- **Local executor**: Configured to run processes on this instance
- **Simple custom pipelines**: Basic Nextflow scripts execute successfully
- **Resource management**: CPU and memory limits configured

### Limitations on SageMaker Notebook Instance ❌

1. **No Docker daemon**: Docker is installed but daemon cannot run without admin access
2. **Conda compatibility issues**: Nextflow 24.04.4 uses `--mkdir` flag incompatible with conda 26.x
3. **S3 access**: Some nf-core test profiles reference private S3 buckets requiring credentials
4. **No Singularity**: Alternative container runtime not available

### For Production nf-core Pipelines, Use:

- **EC2 instance with Docker** - full container support
- **HPC cluster with Slurm + Singularity** - typical academic setup  
- **AWS Batch** - cloud-native distributed execution
- **Seqera Platform** (formerly Tower) - managed Nextflow service

This notebook demonstrates local Nextflow concepts but is not suitable for production nf-core pipelines.

### Install Nextflow on the SageMaker instance

**Note:** Java is already available via conda's openjdk package, so no manual Java installation needed.

In [25]:
# Install Nextflow 24.04.4 (compatible with sarek 3.4.0)
!conda install -c bioconda nextflow=24.04.4 -y

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [6]:
%%sh 
nextflow -version


      N E X T F L O W
      version 26.04.4 build 12445
      created 17-06-2026 16:30 UTC 
      cite doi:10.1038/nbt.3820
      http://nextflow.io



### Configure AWS credentials (SKIP - not needed for local execution)

In [ ]:
# Skip this cell for local execution - AWS credentials not needed
# %env AWS_ACCESS_KEY_ID=""
# %env AWS_SECRET_ACCESS_KEY=""
# %env AWS_SESSION_TOKEN=""

In [29]:
%%sh
# Unset AWS credentials for local execution
unset AWS_ACCESS_KEY_ID
unset AWS_SECRET_ACCESS_KEY
unset AWS_SESSION_TOKEN

### Set up Nextflow config for local execution

**Local vs Cloud Execution:**
- **Local** (`executor = 'local'`): All pipeline processes run on this EC2/SageMaker instance
- **Cloud** (`executor = 'awsbatch'`): Each process runs as a separate AWS Batch job on distributed workers

For this configuration we're using **local** mode.

In [7]:
%%sh
# Create directory in tmp for user
mkdir -p /tmp/$USER 

# Write nextflow config file for LOCAL execution with conda
echo """process {
    executor = 'local'
    cpus = 8
    memory = '30 GB'
}

conda {
    enabled = true
    useMamba = true
    createOptions = '--yes --quiet'
}""" > /tmp/$USER/config

### Run a test pipeline

**Simple custom pipeline** - Successfully demonstrates local execution works.

For nf-core pipelines (sarek, rnaseq, etc.), see limitations above. They require:
- Container runtime (Docker/Singularity) 
- Or compatible conda version
- Proper AWS credentials if using cloud storage

In [8]:
%%sh
cd /tmp/$USER

# Simple test pipeline - WORKS on SageMaker notebook
nextflow run test.nf -c config

echo ""
echo "=========================================="
echo "✅ Local Nextflow execution successful!"
echo "=========================================="


 N E X T F L O W   ~  version 26.04.4

Launching `test.nf` [distraught_carlsson] revision: 4daeced48e

[-        ] sayHello -

executor >  local (1)
[1f/77c9be] sayHello | 1 of 1 ✔
Hello from Nextflow local executor!
CPU: 8
Memory: 30 GB


executor >  local (1)
[1f/77c9be] sayHello | 1 of 1 ✔
Hello from Nextflow local executor!
CPU: 8
Memory: 30 GB



✅ Local Nextflow execution successful!


### Key Learnings from This Session

**Nextflow Execution Modes:**
- **Local** (`executor = 'local'`): Runs all processes on one machine (this instance)
- **AWS Batch** (`executor = 'awsbatch'`): Distributes work across cloud workers
- **Slurm/PBS**: Common on HPC clusters

**Version Pinning:**
- Use `-r <version>` to pin pipeline versions for reproducibility
- Example: `nextflow run nf-core/sarek -r 3.4.0`
- Without `-r`, gets latest release (may have breaking changes)

**Container vs Conda:**
- **Containers (Docker/Singularity)**: Recommended, fully reproducible, isolated
- **Conda**: Fallback when containers unavailable, can have dependency conflicts
- nf-core pipelines are designed for containers first

**SageMaker vs EC2:**
- **SageMaker Notebooks**: Good for Jupyter/Python work, limited for Nextflow
- **EC2 with Docker**: Better for Nextflow, full container support
- **AWS Batch**: Best for production pipelines, auto-scaling

**What You Learned:**
- How to install and configure Nextflow locally
- Local executor configuration (cpus, memory limits)
- Difference between local and cloud execution
- Why production genomics pipelines need proper infrastructure

In [5]:
!nextflow run 1-hello.nf --input 'Hello World!' 


 N E X T F L O W   ~  version 26.04.4

Cannot find script file: 1-hello.nf
